# Proyecto: Análisis de Estacionalidad en Ventas de Automóviles por Marca
**Materia:** Prácticas y Herramientas de Big Data  
**Integrantes:**
* Jose Alexander Quishpe Reinoso
* Julián Garofalo

## Explicación paso a paso
Este notebook está estructurado para llevar a cabo el análisis exploratorio y estadístico del dataset *Car Sales Data*, el cual contiene aproximadamente 2,500,000 de registros. El propósito fundamental es identificar si las ventas y los ingresos de las diferentes marcas de vehículos presentan un comportamiento estacional o si se distribuyen de manera uniforme a lo largo del año fiscal.

El flujo metodológico desarrollado paso a paso comprende:
1. **Extracción y simulación de ingesta:** Configuración del entorno de Big Data en Colab y simulación de consumo de API.
2. **Limpieza y preprocesamiento de millones de filas:** Remoción de duplicados, análisis de nulos y transformaciones de tipos de datos.
3. **Enriquecimiento temporal (Feature Engineering):** Extracción de componentes clave como meses, semanas ISO y trimestres a partir de la fecha.
4. **Formulación de Hipótesis:** Planteamiento formal de los supuestos estacionales de negocio.
5. **Preparación de Estructuras para Modelos:** Organización del espacio para la aplicación de modelos estadísticos y de descomposición de series de tiempo.

---
## Importación de librerías

Antes de comenzar, importamos todas las librerías necesarias para el proyecto. Cada librería cumple un rol específico:

| Librería | Para qué se usa |
|----------|----------------|
| `requests` | Hacer llamadas a la API pública |
| `pandas` | Manipular y limpiar los datos en tablas |
| `numpy` | Operaciones matemáticas y manejo de valores nulos |
| `matplotlib` | Crear gráficos estáticos |
| `seaborn` | Gráficos estadísticos más visuales |
| `sklearn` | Modelos de machine learning |
| `warnings` | Suprimir advertencias innecesarias en la salida |

In [1]:
# ── Llamadas HTTP a APIs externas ──────────────────────────────────
import requests                          # Permite consumir APIs públicas via HTTP

# ── Manipulación de datos ───────────────────────────────────────────
import pandas as pd                      # Tablas de datos (DataFrames)
import numpy as np                       # Operaciones numéricas y manejo de NaN

# ── Visualización ──────────────────────────────────────────────────
import matplotlib.pyplot as plt          # Gráficos básicos (barras, líneas, histogramas)
import seaborn as sns                    # Gráficos estadísticos sobre matplotlib

# ── Preprocesamiento (Scikit-learn) ────────────────────────────────
from sklearn.preprocessing import LabelEncoder        # Convierte texto a números para los modelos
from sklearn.preprocessing import StandardScaler      # Escala variables numéricas a la misma magnitud
from sklearn.model_selection import train_test_split  # Divide datos en entrenamiento y prueba

# ── Modelos de Machine Learning ────────────────────────────────────
from sklearn.linear_model import LinearRegression     # Modelo 1: Regresión Lineal
from sklearn.ensemble import RandomForestRegressor    # Modelo 2: Random Forest
from sklearn.ensemble import GradientBoostingRegressor # Modelo 3: Gradient Boosting

# ── Métricas de evaluación ─────────────────────────────────────────
from sklearn.metrics import mean_absolute_error       # Error absoluto medio
from sklearn.metrics import mean_squared_error        # Error cuadrático medio
from sklearn.metrics import r2_score                  # Coeficiente de determinación R²

# ── Configuración general ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')        # Oculta advertencias para mantener la salida limpia

# Configuración visual de los gráficos
plt.style.use('seaborn-v0_8-whitegrid') # Estilo limpio con cuadrícula para los gráficos
pd.set_option('display.max_columns', None) # Muestra todas las columnas al imprimir el DataFrame

print('✅ Todas las librerías importadas correctamente')

✅ Todas las librerías importadas correctamente


---

## 3. Extracción de Datos

### Explicación técnica
En esta sección se consume la **API pública de Baselight**, que permite acceder al dataset de Kaggle directamente desde Python mediante una consulta SQL, sin necesidad de descargarlo previamente. 

Para cumplir con los estándares de proyectos de Big Data, se plantea una arquitectura híbrida: los metadatos o una muestra controlada pueden consumirse desde esta API pública (utilizando la librería `requests`), mientras que el bloque principal de datos se procesa de forma optimizada desde el almacenamiento local de Colab mediante la carga por bloques o lectura directa con `pandas`.

### Detalles de conexión
* **Endpoint utilizado:** `https://baselight.app/api/v0/query`
* **Método:** `POST`
* **Formato de respuesta:** JSON con claves `rows` y `columns`

### Variables obtenidas del dataset
Al realizar la extracción, se obtienen las siguientes columnas estructuradas:

* `Date`: Fecha exacta de la transacción (clave para el análisis de series temporales).
* `Salesperson`: Nombre del asesor comercial.
* `Customer Name`: Nombre del comprador.
* `Car Make`: Fabricante del vehículo (Toyota, Honda, Ford, Chevrolet, Nissan).
* `Car Model`: Modelo específico del auto.
* `Car Year`: Año de fabricación del vehículo.
* `Sale Price`: Precio de venta final en USD (variable de ingresos).
* `Commission Rate`: Porcentaje de comisión asignado.
* `Commission Earned`: Monto total de comisión generado.

**Cantidad de registros estimados:** 2,500,000 filas.

In [2]:
# ── PASO 1: Definir la URL de la API ───────────────────────────────
# Endpoint público de Baselight para ejecutar consultas SQL sobre datasets de Kaggle
url = 'https://baselight.app/api/v0/query'

# ── PASO 2: Escribir la consulta SQL ───────────────────────────────
# SELECT * trae todas las columnas disponibles
# LIMIT 1000000 limita a 1 millon de filas para no saturar la memoria de Colab
query = """
SELECT * FROM kaggle.suraj520_car_sales_data
LIMIT 10
"""

# ── PASO 3: Hacer la llamada a la API ──────────────────────────────
# requests.post() envía la consulta al servidor
# json={"sql": query} empaca la consulta en formato JSON que la API entiende
response = requests.post(url, json={'sql': query})

# Verificamos que la respuesta fue exitosa (200 = OK)
print('Estado de la API:', response.status_code)
print('Respuesta exitosa' if response.status_code == 200 else 'Error en la solicitud')

Estado de la API: 404
Error en la solicitud


In [ ]:
# ── PASO 4: Convertir la respuesta a DataFrame ─────────────────────
# response.json() convierte el JSON de la API a un diccionario Python
data = response.json()

# pd.DataFrame() crea la tabla usando:
#   data['rows']    → las filas con los datos
#   data['columns'] → los nombres de las columnas
df = pd.DataFrame(data['rows'], columns=data['columns'])

# ── PASO 5: Variables obtenidas ────────────────────────────────────
print('Columnas del dataset:')
print(df.columns.tolist())

# ── PASO 6: Cantidad de registros ──────────────────────────────────
print(f'\n Registros extraídos: {len(df)}')
print(f' Dimensiones (filas x columnas): {df.shape}')

# ── PASO 7: Vista previa del dataset ───────────────────────────────
# head() muestra las primeras 5 filas como evidencia de la extracción
df.head()